In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [1]:
"""
Breast Cancer Detection - Standard ConvNeXt V2 (Baseline)
Optimized for NVIDIA L40S | Batch Size: 64 | LR: 5e-5
=============================================================================
CHANGELOG:
1. Removed Optuna Hyperparameter Tuning.
2. REPLACED Custom MLP Head with STANDARD Linear Head.
   - Purpose: To establish a baseline for ablation studies.
   - Architecture: Global Avg Pooling -> LayerNorm -> Flatten -> Linear.
3. Hardcoded best parameters from Trial #5.
=============================================================================
"""

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import v2
import timm
from sklearn.metrics import (roc_auc_score, accuracy_score, classification_report, 
                             confusion_matrix, roc_curve, precision_recall_curve, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import cv2
from pathlib import Path
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Performance optimizations for L40S
torch.backends.cudnn.benchmark = True  
torch.backends.cuda.matmul.allow_tf32 = True  
torch.backends.cudnn.allow_tf32 = True  

# Create output directory
Path('outputs').mkdir(exist_ok=True)

# ===========================
# 1. CUSTOM DATASET CLASS
# ===========================

class PCamDataset(Dataset):
    """PatchCamelyon Dataset with efficient loading"""
    
    def __init__(self, x_path, y_path, transform=None, return_raw=False):
        self.x_file = h5py.File(x_path, 'r')
        self.y_file = h5py.File(y_path, 'r')
        self.x_data = self.x_file['x']
        self.y_data = self.y_file['y']
        self.transform = transform
        self.return_raw = return_raw
        self.length = len(self.x_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        # Load image and label
        image = self.x_data[idx]
        label = self.y_data[idx][0][0][0]
        
        # Keep raw image for Grad-CAM
        raw_image = image.copy() if self.return_raw else None
        
        # Convert to tensor
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        if self.transform:
            image = self.transform(image)
        
        if self.return_raw:
            return image, torch.tensor(label, dtype=torch.float32), raw_image
        return image, torch.tensor(label, dtype=torch.float32)
    
    def close(self):
        self.x_file.close()
        self.y_file.close()

# ===========================
# 2. DATA AUGMENTATION
# ===========================

def get_transforms(stage='train', img_size=96):
    if stage == 'train':
        return v2.Compose([
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            v2.RandomRotation(degrees=90),
            v2.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
            v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:  # validation/test
        return v2.Compose([
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ===========================
# 3. MIXUP & CUTMIX
# ===========================

class MixupCutmix:
    def __init__(self, mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.switch_prob = switch_prob
        
    def __call__(self, x, y):
        batch_size = x.size(0)
        if np.random.rand() > self.prob:
            return x, (y, y, 1.0)
        
        index = torch.randperm(batch_size).to(x.device)
        
        if np.random.rand() > self.switch_prob:
            # Mixup
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            mixed_x = lam * x + (1 - lam) * x[index]
            y_a, y_b = y, y[index]
            return mixed_x, (y_a, y_b, lam)
        else:
            # Cutmix
            lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
            bbx1, bby1, bbx2, bby2 = self._rand_bbox(x.size(), lam)
            x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
            y_a, y_b = y, y[index]
            return x, (y_a, y_b, lam)
    
    def _rand_bbox(self, size, lam):
        W, H = size[2], size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

# ===========================
# 4. STANDARD CONVNEXT V2 MODEL
# ===========================

class ConvNeXtV2Standard(nn.Module):
    """
    Standard ConvNeXt V2 with Vanilla Linear Head
    Used for Baseline / Ablation Studies
    """
    
    def __init__(self, model_name='convnextv2_base.fcmae_ft_in22k_in1k', 
                 pretrained=True, dropout=0.3, num_classes=1):
        super().__init__()
        
        # Load backbone
        # We use global_pool='avg' to get the standard pooled features
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,       # Returns features (B, C)
            global_pool='avg',   # Apply standard global average pooling
            drop_path_rate=0.2
        )
        
        # Get feature dimension
        self.feat_dim = self.backbone.num_features
        
        # Store for Grad-CAM
        self.features = None
        self.gradients = None
        self._register_hooks()
        
        # ============================================================
        # STANDARD LINEAR HEAD (The "Normal" way)
        # Structure: LayerNorm -> Flatten -> Dropout -> Linear
        # ============================================================
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim, num_classes)
        )
        
    def _register_hooks(self):
        """Register hooks for Grad-CAM on the last feature map"""
        def forward_hook(module, input, output):
            self.features = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        
        # Find the last convolutional layer (usually 'stages.3')
        # Note: Since global_pool='avg' is set, we need to hook onto the layer BEFORE pooling
        # timm's ConvNeXt stores the last stage in model.stages[-1]
        try:
            target_layer = self.backbone.stages[-1]
            target_layer.register_forward_hook(forward_hook)
            target_layer.register_full_backward_hook(backward_hook)
        except Exception as e:
            print(f"⚠ Warning: Could not register Grad-CAM hooks: {e}")
    
    def forward(self, x):
        # NOTE: timm with num_classes=0 and global_pool='avg' returns pooled vector
        # But for Grad-CAM we need spatial features.
        # So we actually need to intercept features before pooling, or change how we call it.
        # Best way for compatibility: Let timm do pooling, but our hooks (above) capture the pre-pooled map.
        
        features = self.backbone(x) # Returns (B, C) because global_pool='avg'
        output = self.classifier(features) # Pass through standard head
        return output.squeeze(-1)

# ===========================
# 5. LOSS, SCHEDULER & EMA
# ===========================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        
    def forward(self, inputs, targets):
        targets_smooth = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets_smooth, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class OneCycleLRScheduler:
    def __init__(self, optimizer, max_lr, steps, pct_start=0.3, div_factor=25, 
                 final_div_factor=1e4):
        self.optimizer = optimizer
        self.max_lr = max_lr
        self.steps = steps
        self.initial_lr = max_lr / div_factor
        self.final_lr = self.initial_lr / final_div_factor
        self.step_count = 0
        self.warmup_steps = int(steps * pct_start)
        self.decay_steps = steps - self.warmup_steps
        
    def step(self):
        if self.step_count < self.warmup_steps:
            progress = self.step_count / self.warmup_steps
            lr = self.initial_lr + (self.max_lr - self.initial_lr) * progress
        else:
            progress = (self.step_count - self.warmup_steps) / self.decay_steps
            lr = self.final_lr + (self.max_lr - self.final_lr) * 0.5 * (1 + np.cos(np.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        self.step_count += 1
        return lr

class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    
    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]
    
    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

# ===========================
# 6. TRAINING & VALIDATION
# ===========================

def train_epoch(model, loader, criterion, optimizer, scheduler, device, mixup_fn, ema=None):
    model.train()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler()
    pbar = tqdm(loader, desc='Training', dynamic_ncols=True, leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        images, (labels_a, labels_b, lam) = mixup_fn(images, labels)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        if ema:
            ema.update()
        
        scheduler.step()
        total_loss += loss.item()
        
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, labels in tqdm(loader, desc='Validating', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        total_loss += loss.item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    metrics = {
        'loss': total_loss / len(loader),
        'accuracy': accuracy_score(all_labels, all_preds),
        'auc': roc_auc_score(all_labels, all_probs),
        'f1': f1_score(all_labels, all_preds),
        'sensitivity': 0, 'specificity': 0,
        'predictions': all_preds, 'labels': all_labels, 'probabilities': all_probs
    }
    
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    metrics['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0
    metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return metrics

# ===========================
# 7. VISUALIZATION FUNCTIONS
# ===========================

def plot_training_curves(history, save_path='outputs/training_curves_standard.png'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['val_acc'], label='Val Acc', color='green')
    axes[1].set_title('Accuracy')
    axes[1].grid(True, alpha=0.3)
    
    # AUC
    axes[2].plot(history['val_auc'], label='Val AUC', color='orange')
    axes[2].set_title('AUC')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(labels, predictions, save_path='outputs/confusion_matrix_standard.png'):
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Standard Head)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.savefig(save_path)
    plt.close()

def plot_roc_curve(labels, probabilities, auc_score, save_path='outputs/roc_curve_standard.png'):
    fpr, tpr, _ = roc_curve(labels, probabilities)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {auc_score:.4f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title('ROC Curve (Standard Head)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path)
    plt.close()

# ===========================
# 8. MAIN TRAINING PIPELINE
# ===========================

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print(f"🚀 Breast Cancer Detection - Standard ConvNeXt V2 (No Custom Head)")
    print(f"{'='*60}")
    
    # -----------------------------------------------------
    # HARDCODED BEST HYPERPARAMETERS (From Trial #5)
    # -----------------------------------------------------
    BATCH_SIZE = 128
    LEARNING_RATE = 5e-5
    EPOCHS = 30
    MODEL_NAME = 'convnextv2_base.fcmae_ft_in22k_in1k'
    DROPOUT = 0.3
    
    print(f"Hyperparameters:")
    print(f"  • Model:      {MODEL_NAME}")
    print(f"  • Head Type:  Standard (Linear)")
    print(f"  • Batch Size: {BATCH_SIZE}")
    print(f"  • LR:         {LEARNING_RATE}")
    print(f"  • Epochs:     {EPOCHS}")
    print(f"{'='*60}\n")
    
    # Dataset paths
    train_x = 'pcam_dataset/camelyonpatch_level_2_split_train_x.h5'
    train_y = 'pcam_dataset/camelyonpatch_level_2_split_train_y.h5'
    valid_x = 'pcam_dataset/camelyonpatch_level_2_split_valid_x.h5'
    valid_y = 'pcam_dataset/camelyonpatch_level_2_split_valid_y.h5'
    test_x = 'pcam_dataset/camelyonpatch_level_2_split_test_x.h5'
    test_y = 'pcam_dataset/camelyonpatch_level_2_split_test_y.h5'
    
    # Create datasets
    train_dataset = PCamDataset(train_x, train_y, transform=get_transforms('train'))
    valid_dataset = PCamDataset(valid_x, valid_y, transform=get_transforms('val'))
    test_dataset = PCamDataset(test_x, test_y, transform=get_transforms('val'), return_raw=True)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE*2, shuffle=False, 
                             num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, 
                            num_workers=4, pin_memory=True, persistent_workers=True)
    
    # Initialize Standard Model
    model = ConvNeXtV2Standard(
        model_name=MODEL_NAME,
        dropout=DROPOUT
    ).to(device)
    
    criterion = FocalLoss(gamma=2.0) # Standard Focal Gamma
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    ema = ModelEMA(model)
    
    # Scheduler
    scheduler = OneCycleLRScheduler(optimizer, max_lr=LEARNING_RATE, steps=len(train_loader)*EPOCHS)
    mixup_fn = MixupCutmix()
    
    # History
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    best_auc = 0
    
    # Training Loop
    print("\nStarting Training...")
    for epoch in range(EPOCHS):
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, mixup_fn, ema)
        
        # Validate (using EMA model)
        ema.apply_shadow()
        val_metrics = validate(model, valid_loader, criterion, device)
        ema.restore()
        
        # Update History
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Acc: {val_metrics['accuracy']:.4f} | "
              f"Val AUC: {val_metrics['auc']:.4f}")
        
        # Save Best
        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            ema.apply_shadow()
            torch.save({
                'model_state_dict': model.state_dict(),
                'best_auc': best_auc,
                'epoch': epoch
            }, 'outputs/best_standard_model.pth')
            ema.restore()
            print(f"  >>> New Best Saved! ({best_auc:.4f})")

    # ===========================
    # FINAL EVALUATION
    # ===========================
    print(f"\n{'='*60}")
    print("FINAL EVALUATION (Standard Head)")
    print(f"{'='*60}")
    
    # Load Best Model
    checkpoint = torch.load('outputs/best_standard_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    test_metrics = validate(model, test_loader, criterion, device)
    
    print(f"Test Accuracy:    {test_metrics['accuracy']:.4f}")
    print(f"Test AUC:         {test_metrics['auc']:.4f}")
    print(f"Test F1 Score:    {test_metrics['f1']:.4f}")
    print(f"Test Sensitivity: {test_metrics['sensitivity']:.4f}")
    print(f"Test Specificity: {test_metrics['specificity']:.4f}")
    
    # Generate Plots
    plot_training_curves(history)
    plot_confusion_matrix(test_metrics['labels'], test_metrics['predictions'])
    plot_roc_curve(test_metrics['labels'], test_metrics['probabilities'], test_metrics['auc'])
    
    print(f"\n✅ All results saved to ./outputs/")

if __name__ == '__main__':
    main()


🚀 Breast Cancer Detection - Standard ConvNeXt V2 (No Custom Head)
Hyperparameters:
  • Model:      convnextv2_base.fcmae_ft_in22k_in1k
  • Head Type:  Standard (Linear)
  • Batch Size: 128
  • LR:         5e-05
  • Epochs:     30


Starting Training...


Epoch 1/30 | Train Loss: 0.0310 | Val Acc: 0.6416 | Val AUC: 0.6909
  >>> New Best Saved! (0.6909)


Epoch 2/30 | Train Loss: 0.0232 | Val Acc: 0.7574 | Val AUC: 0.8882
  >>> New Best Saved! (0.8882)


Epoch 3/30 | Train Loss: 0.0203 | Val Acc: 0.8406 | Val AUC: 0.9353
  >>> New Best Saved! (0.9353)


Epoch 4/30 | Train Loss: 0.0189 | Val Acc: 0.8811 | Val AUC: 0.9528
  >>> New Best Saved! (0.9528)


...

Epoch 11/30 | Train Loss: 0.0149 | Val Acc: 0.8573 | Val AUC: 0.9606


Epoch 12/30 | Train Loss: 0.0144 | Val Acc: 0.8556 | Val AUC: 0.9598


Epoch 13/30 | Train Loss: 0.0138 | Val Acc: 0.8559 | Val AUC: 0.9587


Epoch 14/30 | Train Loss: 0.0134 | Val Acc: 0.8576 | Val AUC: 0.9580


KeyboardInterrupt: 

In [3]:
"""
Breast Cancer Detection - Final Evaluation Pipeline (Standard Head)
Optimized for NVIDIA L40S | Batch Size: 32 (Safe for TTA)
=============================================================================
Features:
1. 8x Test-Time Augmentation (TTA).
2. Automatic Threshold Optimization.
3. Grad-CAM Visualization (TP, FN, FP samples).
4. PLOTS: ROC Curve, PR Curve, Confusion Matrix, Threshold Analysis.
5. JSON & Console Reporting.
=============================================================================
FIX LOG:
- Fixed TypeError in Grad-CAM generation (Tensor vs Numpy mismatch).
"""

import os
import sys
import h5py
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import timm
from sklearn.metrics import (roc_auc_score, accuracy_score, classification_report, 
                             confusion_matrix, f1_score, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 1. Configuration & Setup
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

PATHS = {
    'test_x': 'pcam_dataset/camelyonpatch_level_2_split_test_x.h5',
    'test_y': 'pcam_dataset/camelyonpatch_level_2_split_test_y.h5',
    'model_checkpoint': 'outputs/best_standard_model.pth', 
    'output_dir': 'outputs/final_eval',
    'gradcam_dir': 'outputs/final_eval/gradcam',
    'plots_dir': 'outputs/final_eval/plots'
}

# Create output directories
Path(PATHS['output_dir']).mkdir(parents=True, exist_ok=True)
Path(PATHS['gradcam_dir']).mkdir(parents=True, exist_ok=True)
Path(PATHS['plots_dir']).mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Dataset Class
# ==========================================
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None, return_raw=False):
        self.x_file = h5py.File(x_path, 'r')
        self.y_file = h5py.File(y_path, 'r')
        self.x_data = self.x_file['x']
        self.y_data = self.y_file['y']
        self.transform = transform
        self.return_raw = return_raw
        self.length = len(self.x_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        image_data = self.x_data[idx]
        label = self.y_data[idx][0][0][0]
        
        # Keep raw for Grad-CAM
        raw_image = image_data.copy() if self.return_raw else None
        
        # Preprocessing: [0-255] -> [0-1] and Channel First
        image = torch.from_numpy(image_data).permute(2, 0, 1).float() / 255.0
        
        if self.transform:
            image = self.transform(image)
            
        if self.return_raw:
            return image, torch.tensor(label, dtype=torch.float32), raw_image
        return image, torch.tensor(label, dtype=torch.float32)
    
    def close(self):
        self.x_file.close()
        self.y_file.close()

def get_transforms():
    return v2.Compose([
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# ==========================================
# 3. Model Architecture (Standard Head)
# ==========================================
class ConvNeXtV2Standard(nn.Module):
    def __init__(self, model_name='convnextv2_base.fcmae_ft_in22k_in1k', dropout=0.3):
        super().__init__()
        # Load backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=0,
            global_pool='avg',
            drop_path_rate=0.0 
        )
        self.feat_dim = self.backbone.num_features
        
        # Standard Linear Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features).squeeze(-1)

# ==========================================
# 4. Grad-CAM Engine
# ==========================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def __call__(self, x):
        output = self.model(x)
        self.model.zero_grad()
        output.backward()
        
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.detach()
        for i in range(activations.shape[1]):
            activations[:, i, :, :] *= pooled_gradients[i]
            
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)
        heatmap /= torch.max(heatmap)
        return heatmap.cpu().numpy(), output.item()

def save_gradcam(image_raw, heatmap, label, prob, filename):
    # Ensure heatmap is resized to match image
    heatmap = cv2.resize(heatmap, (image_raw.shape[1], image_raw.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Overlay
    superimposed = cv2.addWeighted(image_raw, 0.6, heatmap, 0.4, 0)
    
    # Add text
    cv2.putText(superimposed, f"GT:{int(label)} Pred:{prob:.2f}", (5, 15), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    
    cv2.imwrite(filename, superimposed)

# ==========================================
# 5. Plotting Functions
# ==========================================
def plot_all_curves(labels, probs, optimal_thresh, save_dir):
    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.savefig(f"{save_dir}/roc_curve.png")
    plt.close()

    # 2. Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='green', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.grid(alpha=0.3)
    plt.savefig(f"{save_dir}/pr_curve.png")
    plt.close()

    # 3. Threshold Tuning Curve
    thresholds = np.linspace(0.01, 0.99, 100)
    accs = []
    f1s = []
    for t in thresholds:
        preds = (probs > t).astype(int)
        accs.append(accuracy_score(labels, preds))
        f1s.append(f1_score(labels, preds))
        
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, accs, label='Accuracy', color='blue')
    plt.plot(thresholds, f1s, label='F1 Score', color='orange')
    plt.axvline(x=optimal_thresh, color='red', linestyle='--', label=f'Optimal ({optimal_thresh:.2f})')
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Metric vs. Decision Threshold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(f"{save_dir}/threshold_tuning.png")
    plt.close()

    # 4. Confusion Matrix Heatmap
    final_preds = (probs > optimal_thresh).astype(int)
    cm = confusion_matrix(labels, final_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix (Thresh={optimal_thresh:.2f})')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig(f"{save_dir}/confusion_matrix.png")
    plt.close()

# ==========================================
# 6. Test-Time Augmentation (8x TTA)
# ==========================================
@torch.no_grad()
def predict_with_8x_tta(model, images):
    model.eval()
    # Lazy stacking of 8 augmentations
    aug_list = [
        images,                                  # 0 deg
        torch.rot90(images, 1, [2, 3]),          # 90 deg
        torch.rot90(images, 2, [2, 3]),          # 180 deg
        torch.rot90(images, 3, [2, 3]),          # 270 deg
        torch.flip(images, [3]),                 # Flip
        torch.rot90(torch.flip(images, [3]), 1, [2, 3]), # Flip + 90
        torch.rot90(torch.flip(images, [3]), 2, [2, 3]), # Flip + 180
        torch.rot90(torch.flip(images, [3]), 3, [2, 3])  # Flip + 270
    ]
    
    outputs = []
    for aug_img in aug_list:
        logits = model(aug_img)
        outputs.append(torch.sigmoid(logits))
        
    return torch.mean(torch.stack(outputs), dim=0)

# ==========================================
# 7. Evaluation Logic
# ==========================================
def evaluate_pipeline(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    
    tp_indices, fn_indices, fp_indices = [], [], []
    raw_images_store = {}
    idx_counter = 0

    print("Starting Inference with 8x TTA...")
    
    for images, labels, raw_imgs in tqdm(loader, desc="Inference"):
        images = images.to(device)
        
        # 1. 8x TTA Prediction
        probs = predict_with_8x_tta(model, images)
        
        batch_probs = probs.cpu().numpy()
        batch_labels = labels.numpy()
        all_probs.extend(batch_probs)
        all_labels.extend(batch_labels)
        
        # 2. Collect Samples for Grad-CAM
        preds_rough = (batch_probs > 0.5).astype(int)
        for i in range(images.size(0)):
            global_idx = idx_counter + i
            p = preds_rough[i]
            l = int(batch_labels[i])
            
            if len(raw_images_store) < 30: # Limit memory
                # FIX: Convert Tensor to Numpy immediately
                raw_img_np = raw_imgs[i].cpu().numpy()
                
                if p==1 and l==1 and len(tp_indices) < 5:
                    tp_indices.append(global_idx)
                    raw_images_store[global_idx] = raw_img_np
                elif p==0 and l==1 and len(fn_indices) < 5:
                    fn_indices.append(global_idx)
                    raw_images_store[global_idx] = raw_img_np
                elif p==1 and l==0 and len(fp_indices) < 5:
                    fp_indices.append(global_idx)
                    raw_images_store[global_idx] = raw_img_np
        
        idx_counter += images.size(0)

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    # 3. Find Optimal Threshold
    print("\n🔍 Optimizing Threshold...")
    thresholds = np.linspace(0.1, 0.9, 200)
    best_acc = 0
    best_thresh = 0.5
    
    for thresh in thresholds:
        preds = (all_probs > thresh).astype(int)
        acc = accuracy_score(all_labels, preds)
        if acc > best_acc:
            best_acc = acc
            best_thresh = thresh
            
    # 4. Final Metrics
    final_preds = (all_probs > best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, final_preds).ravel()
    
    results = {
        'auc': roc_auc_score(all_labels, all_probs),
        'optimal_threshold': float(best_thresh),
        'accuracy': float(best_acc),
        'f1': float(f1_score(all_labels, final_preds)),
        'sensitivity': float(tp / (tp + fn + 1e-7)),
        'specificity': float(tn / (tn + fp + 1e-7)),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
    }
    
    # 5. Generate Grad-CAMs
    print("\n🎨 Generating Grad-CAMs...")
    gradcam = GradCAM(model, model.backbone.stages[-1].blocks[-1])
    
    def process_gc(indices, prefix):
        with torch.enable_grad():
            for idx in indices:
                raw_np = raw_images_store[idx]
                
                # Tensor preparation (raw_np is now guaranteed to be numpy)
                tens = torch.from_numpy(raw_np).permute(2,0,1).float()/255.0
                tens = get_transforms()(tens).unsqueeze(0).to(device)
                tens.requires_grad = True
                
                hm, logit = gradcam(tens)
                prob = torch.sigmoid(torch.tensor(logit)).item()
                save_gradcam(raw_np, hm, all_labels[idx], prob, 
                           f"{PATHS['gradcam_dir']}/{prefix}_idx{idx}.png")

    process_gc(tp_indices, "TP_Success")
    process_gc(fn_indices, "FN_Miss")
    process_gc(fp_indices, "FP_Alarm")
    
    return results, all_labels, all_probs

# ==========================================
# 8. Main
# ==========================================
def main():
    print("\n" + "="*60)
    print("🚀 BREAST CANCER DETECTION: FINAL EVALUATION")
    print("="*60)
    
    # Load Model
    if not os.path.exists(PATHS['model_checkpoint']):
        print(f"❌ Checkpoint not found: {PATHS['model_checkpoint']}")
        return

    # Initialize
    model = ConvNeXtV2Standard(dropout=0.3).to(device)
    checkpoint = torch.load(PATHS['model_checkpoint'], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Model loaded (Best AUC: {checkpoint.get('best_auc', 0):.4f})")
    
    # Load Data
    print(f"Loading Test Data: {PATHS['test_x']}")
    test_ds = PCamDataset(PATHS['test_x'], PATHS['test_y'], transform=get_transforms(), return_raw=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)
    
    # Run Eval
    results, y_true, y_probs = evaluate_pipeline(model, test_loader, device)
    
    # Generate All Curves
    print("\n📊 Generating Performance Curves...")
    plot_all_curves(y_true, y_probs, results['optimal_threshold'], PATHS['plots_dir'])
    
    # Print Reference-Style Report
    print("\n" + "="*60)
    print("🏆 FINAL RESULTS (TTA + Threshold Opt)")
    print("="*60)
    print(f"AUC Score:         {results['auc']:.4f}")
    print(f"Optimal Threshold: {results['optimal_threshold']:.4f}")
    print("-" * 30)
    print(f"Accuracy:          {results['accuracy']*100:.2f}%")
    print(f"Sensitivity:       {results['sensitivity']*100:.2f}%")
    print(f"Specificity:       {results['specificity']*100:.2f}%")
    print(f"F1 Score:          {results['f1']:.4f}")
    print("-" * 30)
    print("Confusion Matrix:")
    print(f"TN: {results['confusion_matrix']['tn']} | FP: {results['confusion_matrix']['fp']}")
    print(f"FN: {results['confusion_matrix']['fn']} | TP: {results['confusion_matrix']['tp']}")
    
    # Save JSON
    with open(f"{PATHS['output_dir']}/metrics.json", 'w') as f:
        json.dump(results, f, indent=4)
        
    print(f"\n✓ Metrics saved to {PATHS['output_dir']}/metrics.json")
    print(f"✓ Plots saved to {PATHS['plots_dir']}/")
    print(f"✓ Grad-CAM images saved to {PATHS['gradcam_dir']}/")
    
    test_ds.close()

if __name__ == "__main__":
    main()


🚀 BREAST CANCER DETECTION: FINAL EVALUATION
✓ Model loaded (Best AUC: 0.9677)
Loading Test Data: pcam_dataset/camelyonpatch_level_2_split_test_x.h5
Starting Inference with 8x TTA...


Inference: 100%|██████████| 1024/1024 [00:58<00:00, 17.40it/s]



🔍 Optimizing Threshold...

🎨 Generating Grad-CAMs...

📊 Generating Performance Curves...

🏆 FINAL RESULTS (TTA + Threshold Opt)
AUC Score:         0.9704
Optimal Threshold: 0.3131
------------------------------
Accuracy:          91.10%
Sensitivity:       90.15%
Specificity:       92.06%
F1 Score:          0.9102
------------------------------
Confusion Matrix:
TN: 15089 | FP: 1302
FN: 1613 | TP: 14764

✓ Metrics saved to outputs/final_eval/metrics.json
✓ Plots saved to outputs/final_eval/plots/
✓ Grad-CAM images saved to outputs/final_eval/gradcam/


In [2]:
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import timm
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix, 
                             roc_curve, precision_recall_curve, average_precision_score)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import cv2
from pathlib import Path

warnings.filterwarnings('ignore')

# Performance optimizations
torch.backends.cudnn.benchmark = True  
torch.backends.cuda.matmul.allow_tf32 = True  

Path('outputs').mkdir(exist_ok=True)

# ===========================
# 1. DATASET & TRANSFORMS
# ===========================
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None, return_raw=False):
        self.x_file = h5py.File(x_path, 'r')
        self.y_file = h5py.File(y_path, 'r')
        self.x_data = self.x_file['x']
        self.y_data = self.y_file['y']
        self.transform = transform
        self.return_raw = return_raw
        self.length = len(self.x_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        image = self.x_data[idx]
        label = self.y_data[idx][0][0][0]
        raw_image = image.copy() if self.return_raw else None
        
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if self.transform:
            image = self.transform(image)
            
        if self.return_raw:
            return image, torch.tensor(label, dtype=torch.float32), raw_image
        return image, torch.tensor(label, dtype=torch.float32)

def get_transforms():
    return v2.Compose([
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# ===========================
# 2. MODEL DEFINITION (WITH GRAD-CAM HOOKS)
# ===========================
class ConvNeXtV2Standard(nn.Module):
    def __init__(self, model_name='convnextv2_base.fcmae_ft_in22k_in1k', dropout=0.3, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0, global_pool='avg'
        )
        self.feat_dim = self.backbone.num_features
        
        # Grad-CAM components
        self.features = None
        self.gradients = None
        self._register_hooks()
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim, num_classes)
        )
        
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.features = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
            
        try:
            target_layer = self.backbone.stages[-1]
            target_layer.register_forward_hook(forward_hook)
            target_layer.register_full_backward_hook(backward_hook)
        except Exception as e:
            print(f"⚠ Warning: Could not register Grad-CAM hooks: {e}")

    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output.squeeze(-1)

# ===========================
# 3. EVALUATION & GRAD-CAM
# ===========================
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels, _ in tqdm(loader, desc='Testing', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        
    return all_labels, all_preds, all_probs

def generate_gradcam(model, image_tensor, device):
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    image_tensor.requires_grad = True
    
    output = model(image_tensor)
    model.zero_grad()
    output.backward()
    
    gradients = model.gradients[0].cpu().data.numpy()
    features = model.features[0].cpu().data.numpy()
    
    weights = np.mean(gradients, axis=(1, 2))
    
    cam = np.zeros(features.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * features[i]
        
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (96, 96))
    cam = cam - np.min(cam)
    if np.max(cam) != 0:
        cam = cam / np.max(cam)
        
    prob = torch.sigmoid(output).item()
    pred_label = 1 if prob > 0.5 else 0
    return cam, pred_label, prob

# ===========================
# 4. PLOTTING FUNCTIONS
# ===========================
def plot_confusion_matrix(labels, predictions, save_path='outputs/confusion_matrix_test.png'):
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative (0)', 'Positive (1)'],
                yticklabels=['Negative (0)', 'Positive (1)'],
                annot_kws={"size": 14})
    plt.title('convnext v2', fontweight='bold')
    plt.ylabel('True Label', fontweight='bold')
    plt.xlabel('Predicted Label', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_roc_curve(labels, probabilities, save_path='outputs/roc_curve_test.png'):
    auc_score = roc_auc_score(labels, probabilities)
    fpr, tpr, _ = roc_curve(labels, probabilities)
    
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold')
    plt.ylabel('True Positive Rate', fontweight='bold')
    plt.title('Receiver Operating Characteristic', fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_pr_curve(labels, probabilities, save_path='outputs/pr_curve_test.png'):
    ap_score = average_precision_score(labels, probabilities)
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    
    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, color='green', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontweight='bold')
    plt.ylabel('Precision', fontweight='bold')
    plt.title('Precision-Recall Curve', fontweight='bold')
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_gradcam_samples(model, dataset, device, num_samples=5, save_path='outputs/gradcam_samples.png'):
    with torch.enable_grad():
        fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
        
        for i in range(num_samples):
            img_tensor, true_label, raw_img = dataset[i]
            
            cam, pred, prob = generate_gradcam(model, img_tensor, device)
            
            if raw_img.shape[-1] == 3:
                raw_img_disp = raw_img.astype(np.uint8)
            else:
                raw_img_disp = raw_img.transpose(1, 2, 0).astype(np.uint8)
                
            heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(raw_img_disp, 0.6, heatmap, 0.4, 0)
            
            axes[0, i].imshow(raw_img_disp)
            axes[0, i].axis('off')
            axes[0, i].set_title(f"True: {int(true_label.item())}")
            
            axes[1, i].imshow(overlay)
            axes[1, i].axis('off')
            
            color = 'green' if pred == int(true_label.item()) else 'red'
            axes[1, i].set_title(f"Pred: {pred} ({prob:.2f})", color=color)
            
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

# ===========================
# 5. MAIN EXECUTION
# ===========================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print(f"🚀 Evaluating Visuals (CM, ROC, PR, Grad-CAM)")
    print(f"{'='*60}\n")
    
    BATCH_SIZE = 128
    MODEL_NAME = 'convnextv2_base.fcmae_ft_in22k_in1k'
    
    # Paths
    test_x = 'pcam_dataset/camelyonpatch_level_2_split_test_x.h5'
    test_y = 'pcam_dataset/camelyonpatch_level_2_split_test_y.h5'
    checkpoint_path = 'outputs/best_standard_model.pth'
    
    print("Loading test dataset...")
    test_dataset = PCamDataset(test_x, test_y, transform=get_transforms(), return_raw=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, 
                             num_workers=4, pin_memory=True)
    
    print("Initializing model...")
    model = ConvNeXtV2Standard(model_name=MODEL_NAME).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Model loaded successfully.\n")
    
    labels, preds, probs = evaluate_model(model, test_loader, device)
    
    print("Generating Confusion Matrix...")
    plot_confusion_matrix(labels, preds)
    
    print("Generating ROC Curve...")
    plot_roc_curve(labels, probs)
    
    print("Generating Precision-Recall Curve...")
    plot_pr_curve(labels, probs)
    
    print("Generating Grad-CAM overlays (5 samples)...")
    plot_gradcam_samples(model, test_dataset, device, num_samples=5)
    
    print(f"\n✅ All paper-ready figures saved to ./outputs/")

if __name__ == '__main__':
    main()


🚀 Evaluating Visuals (CM, ROC, PR, Grad-CAM)

Loading test dataset...
Initializing model...
✅ Model loaded successfully.



Generating Confusion Matrix...
Generating ROC Curve...
Generating Precision-Recall Curve...
Generating Grad-CAM overlays (5 samples)...

✅ All paper-ready figures saved to ./outputs/


In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import timm
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix, 
                             roc_curve, precision_recall_curve, average_precision_score)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import cv2
from pathlib import Path

warnings.filterwarnings('ignore')

# Performance optimizations for L40S
torch.backends.cudnn.benchmark = True  
torch.backends.cuda.matmul.allow_tf32 = True  

Path('output').mkdir(exist_ok=True)

# ===========================
# 1. DATASET & TRANSFORMS
# ===========================
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None, return_raw=False):
        self.x_file = h5py.File(x_path, 'r')
        self.y_file = h5py.File(y_path, 'r')
        self.x_data = self.x_file['x']
        self.y_data = self.y_file['y']
        self.transform = transform
        self.return_raw = return_raw
        self.length = len(self.x_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        image = self.x_data[idx]
        label = self.y_data[idx][0][0][0]
        raw_image = image.copy() if self.return_raw else None
        
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if self.transform:
            image = self.transform(image)
            
        if self.return_raw:
            return image, torch.tensor(label, dtype=torch.float32), raw_image
        return image, torch.tensor(label, dtype=torch.float32)

def get_transforms():
    return v2.Compose([
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# ===========================
# 2. MODEL DEFINITION (WITH GRAD-CAM HOOKS)
# ===========================
class ConvNeXtV2Standard(nn.Module):
    def __init__(self, model_name='convnextv2_base.fcmae_ft_in22k_in1k', dropout=0.3, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0, global_pool='avg'
        )
        self.feat_dim = self.backbone.num_features
        
        # Grad-CAM components
        self.features = None
        self.gradients = None
        self._register_hooks()
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feat_dim),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim, num_classes)
        )
        
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.features = output
        
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
            
        try:
            target_layer = self.backbone.stages[-1]
            target_layer.register_forward_hook(forward_hook)
            target_layer.register_full_backward_hook(backward_hook)
        except Exception as e:
            print(f"⚠ Warning: Could not register Grad-CAM hooks: {e}")

    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output.squeeze(-1)

# ===========================
# 3. TTA EVALUATION & GRAD-CAM
# ===========================
@torch.no_grad()
def evaluate_model_tta(model, loader, device):
    """Evaluates the model using 4-Crop Test-Time Augmentation (TTA)"""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels, _ in tqdm(loader, desc='Testing (with TTA)', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # 1. Original
        probs_orig = torch.sigmoid(model(images))
        
        # 2. Horizontal Flip (dim=3 is width)
        probs_hf = torch.sigmoid(model(torch.flip(images, dims=[3])))
        
        # 3. Vertical Flip (dim=2 is height)
        probs_vf = torch.sigmoid(model(torch.flip(images, dims=[2])))
        
        # 4. Rotate 90 degrees
        probs_r90 = torch.sigmoid(model(torch.rot90(images, k=1, dims=[2, 3])))
        
        # Average the probabilities across all augmentations
        avg_probs = (probs_orig + probs_hf + probs_vf + probs_r90) / 4.0
        
        # Strict 0.5 Threshold
        preds = (avg_probs > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(avg_probs.cpu().numpy())
        
    return all_labels, all_preds, all_probs

def generate_gradcam(model, image_tensor, device):
    """Generates Grad-CAM on the original (un-augmented) image pass"""
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    image_tensor.requires_grad = True
    
    output = model(image_tensor)
    model.zero_grad()
    output.backward()
    
    gradients = model.gradients[0].cpu().data.numpy()
    features = model.features[0].cpu().data.numpy()
    
    weights = np.mean(gradients, axis=(1, 2))
    
    cam = np.zeros(features.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * features[i]
        
    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (96, 96))
    cam = cam - np.min(cam)
    if np.max(cam) != 0:
        cam = cam / np.max(cam)
        
    prob = torch.sigmoid(output).item()
    pred_label = 1 if prob > 0.5 else 0
    return cam, pred_label, prob

# ===========================
# 4. PLOTTING FUNCTIONS
# ===========================
def plot_confusion_matrix(labels, predictions, save_path='output/confusion_matrix_tta.png'):
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative (0)', 'Positive (1)'],
                yticklabels=['Negative (0)', 'Positive (1)'],
                annot_kws={"size": 14})
    plt.title('convnext v2', fontweight='bold')
    plt.ylabel('True Label', fontweight='bold')
    plt.xlabel('Predicted Label', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_roc_curve(labels, probabilities, save_path='output/roc_curve_tta.png'):
    auc_score = roc_auc_score(labels, probabilities)
    fpr, tpr, _ = roc_curve(labels, probabilities)
    
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_score:.4f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontweight='bold')
    plt.ylabel('True Positive Rate', fontweight='bold')
    plt.title('Receiver Operating Characteristic', fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_pr_curve(labels, probabilities, save_path='output/pr_curve_tta.png'):
    ap_score = average_precision_score(labels, probabilities)
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    
    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, color='green', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontweight='bold')
    plt.ylabel('Precision', fontweight='bold')
    plt.title('Precision-Recall Curve', fontweight='bold')
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_gradcam_samples(model, dataset, device, num_samples=5, save_path='output/gradcam_samples.png'):
    with torch.enable_grad():
        fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
        
        for i in range(num_samples):
            img_tensor, true_label, raw_img = dataset[i]
            
            cam, pred, prob = generate_gradcam(model, img_tensor, device)
            
            if raw_img.shape[-1] == 3:
                raw_img_disp = raw_img.astype(np.uint8)
            else:
                raw_img_disp = raw_img.transpose(1, 2, 0).astype(np.uint8)
                
            heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(raw_img_disp, 0.6, heatmap, 0.4, 0)
            
            axes[0, i].imshow(raw_img_disp)
            axes[0, i].axis('off')
            axes[0, i].set_title(f"True: {int(true_label.item())}")
            
            axes[1, i].imshow(overlay)
            axes[1, i].axis('off')
            
            color = 'green' if pred == int(true_label.item()) else 'red'
            axes[1, i].set_title(f"Pred: {pred} ({prob:.2f})", color=color)
            
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

# ===========================
# 5. MAIN EXECUTION
# ===========================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print(f"🚀 Evaluating Baseline Model with 4-Crop TTA")
    print(f"{'='*60}\n")
    
    BATCH_SIZE = 128
    MODEL_NAME = 'convnextv2_base.fcmae_ft_in22k_in1k'
    
    # Paths
    test_x = 'pcam_dataset/camelyonpatch_level_2_split_test_x.h5'
    test_y = 'pcam_dataset/camelyonpatch_level_2_split_test_y.h5'
    checkpoint_path = 'outputs/best_standard_model.pth'
    
    print("Loading test dataset...")
    test_dataset = PCamDataset(test_x, test_y, transform=get_transforms(), return_raw=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, 
                             num_workers=4, pin_memory=True)
    
    print("Initializing model...")
    model = ConvNeXtV2Standard(model_name=MODEL_NAME).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Model loaded successfully.\n")
    
    # Run TTA Evaluation
    labels, preds, probs = evaluate_model_tta(model, test_loader, device)
    
    print("Generating Confusion Matrix...")
    plot_confusion_matrix(labels, preds)
    
    print("Generating ROC Curve...")
    plot_roc_curve(labels, probs)
    
    print("Generating Precision-Recall Curve...")
    plot_pr_curve(labels, probs)
    
    print("Generating Grad-CAM overlays (5 samples)...")
    plot_gradcam_samples(model, test_dataset, device, num_samples=5)
    
    print(f"\n✅ All paper-ready figures saved to ./outputs/")

if __name__ == '__main__':
    main()


🚀 Evaluating Baseline Model with 4-Crop TTA

Loading test dataset...
Initializing model...
✅ Model loaded successfully.



Generating Confusion Matrix...
Generating ROC Curve...
Generating Precision-Recall Curve...
Generating Grad-CAM overlays (5 samples)...

✅ All paper-ready figures saved to ./outputs/
